# Lesson 5a — Nonlinear Curve Fitting — Solutions

> **Teacher reference** — This notebook contains worked solutions. Do not share with students.


One of three independent Lesson 5 modules — **5a fitting · 5b statistics · 5c automation** — do them in any order.

This notebook introduces **nonlinear curve fitting with SciPy**: modelling a stress-strain curve with a Voce-type saturation law, estimating parameters and their uncertainties, and checking the fit quality with residuals.

In [ ]:
# Setup cell: run this once at the beginning.
# It installs SciPy (used in this notebook). This is plumbing — just run it.
%pip install -q scipy
print("Setup complete.")

## Nonlinear fitting with SciPy

In Lesson 4 we fitted a **straight line** to the initial elastic region of a stress-strain curve.

But a real stress-strain curve is not straight: after the elastic part it bends over as the material hardens, reaches a maximum (the **ultimate tensile strength**, UTS), and then falls as the specimen necks.

Here we model the **hardening region, up to the UTS**, of specimen **S03** with a simple Voce-type saturation law:

$$
\sigma(\epsilon) = \sigma_s - (\sigma_s - \sigma_0)\,\exp\!\left(-\frac{\epsilon}{\epsilon_c}\right)
$$

- $\sigma_s$ — saturation stress (the plateau the curve heads toward);
- $\sigma_0$ — model stress at zero strain;
- $\epsilon_c$ — how fast the curve approaches saturation.

In [ ]:
# Part A — imports and data loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def load_table(name):
    """Robust CSV loader: works locally and in the browser (JupyterLite).
    Do not worry about the details — this is just plumbing."""
    for folder in ("data", "../data", "."):
        try:
            return pd.read_csv(f"{folder}/{name}")
        except Exception:
            pass
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))

tensile = load_table("tensile_raw.csv")
print(tensile.shape)
tensile.head()

In [ ]:
# Engineering stress and strain from the raw quantities
tensile["area_mm2"] = np.pi * (tensile["diameter_mm"] / 2) ** 2
tensile["stress_MPa"] = tensile["force_N"] / tensile["area_mm2"]
tensile["strain"] = tensile["displacement_mm"] / tensile["gauge_length_mm"]

# Pick one specimen and keep only the hardening region, up to the peak (UTS)
sample_id = "S03"
sample = tensile[tensile["sample_id"] == sample_id].copy().reset_index(drop=True)

peak_idx = sample["stress_MPa"].idxmax()          # row of the ultimate tensile strength
hardening = sample.loc[:peak_idx]                 # everything up to and including the peak

print(f"{sample_id}: {len(sample)} points total, peak at point {peak_idx} "
      f"(UTS = {sample.loc[peak_idx, 'stress_MPa']:.0f} MPa)")
print(f"Fitting the first {len(hardening)} points (up to the UTS).")

In [ ]:
# Visual inspection: full curve, with the region we will fit highlighted
plt.figure(figsize=(6, 4))
plt.plot(sample["strain"], sample["stress_MPa"], "o", markersize=3, color="0.7", label="full curve")
plt.plot(hardening["strain"], hardening["stress_MPa"], "o", markersize=4, label="fit region (up to UTS)")
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Stress-strain curve: {sample_id}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Define the Voce-type model as a Python function
def voce_model(strain, sigma_s, sigma_0, epsilon_c):
    return sigma_s - (sigma_s - sigma_0) * np.exp(-strain / epsilon_c)

# curve_fit needs a starting guess p0 for a nonlinear model.
# Rough reading off the plot above: saturates near ~500 MPa, starts near 0, knee around 0.01.
p0 = [500, 10, 0.01]
print("Initial guesses -> sigma_s =", p0[0], "MPa, sigma_0 =", p0[1], "MPa, epsilon_c =", p0[2])

In [ ]:
# Run the nonlinear fit on the hardening region
x = hardening["strain"].to_numpy()
y = hardening["stress_MPa"].to_numpy()

popt, pcov = curve_fit(voce_model, x, y, p0=p0, maxfev=10000)
sigma_s, sigma_0, epsilon_c = popt

# Parameter uncertainties: square roots of the diagonal of the covariance matrix
sigma_s_err, sigma_0_err, epsilon_c_err = np.sqrt(np.diag(pcov))

# Goodness of fit: coefficient of determination R^2
y_fit = voce_model(x, *popt)
residuals = y - y_fit
r_squared = 1 - np.sum(residuals**2) / np.sum((y - np.mean(y))**2)

print("Fitted parameters (value +/- uncertainty):")
print(f"  sigma_s   = {sigma_s:7.2f} +/- {sigma_s_err:.2f} MPa")
print(f"  sigma_0   = {sigma_0:7.2f} +/- {sigma_0_err:.2f} MPa")
print(f"  epsilon_c = {epsilon_c:7.4f} +/- {epsilon_c_err:.4f}")
print(f"  R^2       = {r_squared:.4f}")

In [ ]:
# Plot the data and the fitted curve
x_smooth = np.linspace(x.min(), x.max(), 300)

plt.figure(figsize=(6, 4))
plt.plot(sample["strain"], sample["stress_MPa"], "o", markersize=3, color="0.7", label="full curve")
plt.plot(x, y, "o", markersize=4, label="fit region")
plt.plot(x_smooth, voce_model(x_smooth, *popt), "-", linewidth=2, color="red", label="Voce fit")
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Nonlinear fit of the hardening region: {sample_id}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Estimated saturation stress: sigma_s = {sigma_s:.0f} +/- {sigma_s_err:.0f} MPa")

## Optional: residuals

Residuals are the differences between measured and fitted values:

```text
residual = data - fit
```

If the model is appropriate, residuals scatter around zero without a strong systematic pattern.

In [ ]:
# Optional residual plot
plt.figure(figsize=(6, 3.5))
plt.axhline(0, linestyle="--", linewidth=1, color="0.4")
plt.plot(x, residuals, "o", markersize=4)
plt.xlabel("Strain [-]")
plt.ylabel("Residual [MPa]")
plt.title(f"Residuals of the Voce fit: {sample_id}")
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 1 — Fit another specimen

Repeat the Voce fit for a different specimen — try `S01` (annealed) or `S04` (aged).

1. select that specimen and keep only its hardening region, up to the peak (UTS);
2. run `curve_fit` with the same model and the same starting guess `p0`;
3. print the saturation stress σ_s with its uncertainty, and the R².

**Check:** is σ_s higher or lower than the quenched `S03`? Is the fit quality (R²) similar?

In [ ]:
# Exercise 1 — SOLUTION

other_id = "S01"
other = tensile[tensile["sample_id"] == other_id].copy().reset_index(drop=True)
other_peak = other["stress_MPa"].idxmax()
other_hardening = other.loc[:other_peak]

x_o = other_hardening["strain"].to_numpy()
y_o = other_hardening["stress_MPa"].to_numpy()

popt_o, pcov_o = curve_fit(voce_model, x_o, y_o, p0=p0, maxfev=10000)
sigma_s_o = popt_o[0]
sigma_s_o_err = np.sqrt(np.diag(pcov_o))[0]

y_fit_o = voce_model(x_o, *popt_o)
r2_o = 1 - np.sum((y_o - y_fit_o)**2) / np.sum((y_o - np.mean(y_o))**2)

print(f"sigma_s for {other_id}: {sigma_s_o:.2f} +/- {sigma_s_o_err:.2f} MPa")
print(f"R^2: {r2_o:.4f}")
# S01 (annealed) sigma_s ~330 MPa < S03 (quenched) ~494 MPa — annealing reduces strength.
# Fit quality is similar (R^2 ~ 0.95).


### Exercise 2 — Use the model to predict

A fitted model is useful because it lets you **predict** values you did not measure directly.
Using the parameters fitted for `S03` (the array `popt` from the fit above):

1. predict the stress at a strain of `0.05` by calling `voce_model(0.05, *popt)`;
2. print the predicted stress;
3. compare it with the saturation stress σ_s (the first element of `popt`).

**Check:** is the predicted stress below σ_s? Write one short comment on why that makes sense.

In [ ]:
# Exercise 2 — SOLUTION

predicted_stress = voce_model(0.05, *popt)
sigma_s_fitted = popt[0]

print(f"Predicted stress at strain = 0.05: {predicted_stress:.1f} MPa")
print(f"Saturation stress sigma_s:         {sigma_s_fitted:.1f} MPa")

# Yes, the predicted stress is below sigma_s.
# The Voce model saturates asymptotically toward sigma_s as strain increases;
# at strain = 0.05 the curve is still rising toward the plateau, not yet there.


## Closing — Course recap

This final lesson ties together the whole course:

```text
L1  ->  Foundations & first look
        variables, types, lists, conditions, loops, functions; a read->compute->plot demo

L2  ->  Python basics (hands-on)
        arithmetic, booleans, if/elif, lists, for, functions, dictionaries, basic plots

L3  ->  From raw files to clean data tables
        read_csv, inspect, filter, calculated columns, cleaning, groupby, save

L4  ->  Plotting and data analysis
        line/scatter/hist/box/bar plots, summary metrics, a linear fit

L5  ->  Three practical tools
        nonlinear fitting, uncertainty & statistics, automation over many files
```

The central message:

```text
Python pays off when the analysis must be repeated, checked, modified, or shared.
```

## Where to go next

Three useful directions after this introductory course:

## 1. SciPy
Fitting, optimization, statistics, interpolation, integration, signal processing.
Start with: `scipy.optimize.curve_fit`, `scipy.stats`, interpolation.

## 2. pandas
Tabular data analysis.
Start with: missing values, merging tables, grouping/aggregation, reshaping.

## 3. Matplotlib
Publication-quality figures.
Start with: `fig, ax = plt.subplots()`, axis customization, saving vector graphics, multi-panel figures.

Final practical advice:

```text
Start with small notebooks that solve real problems from your own research.
```